In [0]:
# import neceassary spark modules
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

**Silver** **Brands**

In [0]:

# Show bronze brands data  
df_brands = spark.table(f'{catalog_name}.bronze.brz_brands')
df_brands.show(10)

In [0]:
# trim column to remove empty spaces
df_silver = df_brands.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
# Remove special charcters
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
# List unique category_code 
df_silver.select('category_code').distinct().show(10)


In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(anomalies, subset='category_code')
df_silver.show(100)

In [0]:
# List unique category_code 
df_silver.select('category_code').distinct().show(10)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog_name}.silver.slv_brands')

**Silver Category**

In [0]:
# Show bronze category data
df_category = spark.table(f'{catalog_name}.bronze.brz_category')
df_category.show(100)

In [0]:
# make column 'category_code' capital
df_category = df_category.withColumn('category_code', F.upper(F.col('category_code')))
df_category.show(100)
 

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_category)
df_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog_name}.silver.slv_category')

**Silver Producs**

In [0]:
# Show bronze products data
df_products = spark.table(f'{catalog_name}.bronze.brz_products')
# df_products.show(100)
display(df_products)

In [0]:
# Make colmuns 'category_code' and 'brand_code' capital
df_products = df_products.withColumn('category_code', F.upper(F.col('category_code')))
df_products = df_products.withColumn('brand_code', F.upper(F.col('brand_code')))
df_products.show(100)


In [0]:
# Convert colmun 'length_cm' to float
df_products = df_products.withColumn('length_cm', F.regexp_replace(F.col('length_cm'), ',', '.'))
df_products = df_products.withColumn('length_cm', F.col('length_cm').cast('float'))
df_products.show(100)

In [0]:
# Convert colmun 'weight_grams' to float
df_products = df_products.withColumn('weight_grams', F.regexp_replace(F.col('weight_grams'), 'g', ''))
df_products = df_products.withColumn('weight_grams', F.col('weight_grams').cast('float'))
df_products.show(100)


In [0]:
# Replace all the NULL value in column 'color' with 'unknown'
df_products = df_products.withColumn('color', F.when(F.col('color').isNull(), 'unknown').otherwise(F.col('color')))
df_products.show(100)